First we need to analyze what predictor variables are significant,(not analyzing name, id, and its already grouped by coverage). also not using risk_tier becuase thats applied after underwriting. Model trained only on data morked training(not holdout)

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
PROC = ROOT / "data" / "processed"

freq   = pd.read_csv(PROC / "frequency_data.csv")   # 40,000 student-coverage rows
claims = pd.read_csv(PROC / "claims.csv")           # 1,819 individual claims

COVERAGES = ['Personal Property', 'Additional Living Expense',
             'Guest Medical', 'Liability']

# training only; holdout is sealed until validation
fq_train = {c: freq[(freq.coverage == c) &
                    (freq.training_holdout == 'training')].reset_index(drop=True)
            for c in COVERAGES}

cl_train = {c: claims[(claims.coverage == c) &
                      (claims.training_holdout == 'training')].reset_index(drop=True)
            for c in COVERAGES}

RATING_VARS = ['class', 'study', 'gpa', 'greek',
               'off_campus', 'distance_to_campus', 'sprinklered']

pd.DataFrame({
    'exposures': {c: len(fq_train[c]) for c in COVERAGES},
    'claims':    {c: len(cl_train[c]) for c in COVERAGES},
    'frequency': {c: fq_train[c].claim_count.mean().round(4) for c in COVERAGES},
})

freq   = pd.read_csv(PROC / "frequency_data.csv")
claims = pd.read_csv(PROC / "claims.csv")

freq   = freq.rename(columns={'class': 'class_year'})
claims = claims.rename(columns={'class': 'class_year'})

seperated training and holdout. renamed class to classyear to avoid python conflicts with class

Personal property

In [9]:
pp = fq_train['Personal Property'][RATING_VARS + ['claim_count']].copy()

for c in ['class_year', 'study', 'greek', 'off_campus']:
    pp[c] = pp[c].astype('category')

pp.head(10)

KeyError: 'class_year'

got relevant data and added 

In [6]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# holdout frame for Personal Property, same treatment as the training frame
pp_hold = freq[(freq.coverage == 'Personal Property') &
               (freq.training_holdout == 'holdout')].copy()
for c in ['class', 'study', 'greek', 'off_campus']:
    pp_hold[c] = pp_hold[c].astype('category')


def poisson_deviance(y, mu):
    """2 * sum[ y*log(y/mu) - (y - mu) ], with 0*log(0) = 0."""
    y  = np.asarray(y,  dtype=float)
    mu = np.asarray(mu, dtype=float)
    term = np.where(y > 0, y * np.log(y / mu), 0.0)
    return 2.0 * np.sum(term - (y - mu))


rows = []

# baseline: intercept only
null = smf.glm('claim_count ~ 1', data=pp, family=sm.families.Poisson()).fit()
rows.append(('(null)', null.aic,
             poisson_deviance(pp_hold.claim_count, null.predict(pp_hold))))

# one model per candidate predictor
for v in RATING_VARS:
    m = smf.glm(f'claim_count ~ {v}', data=pp, family=sm.families.Poisson()).fit()
    rows.append((v, m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step1 = (pd.DataFrame(rows, columns=['predictor', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step1.round(3)

C:\Users\dalto\AppData\Local\Temp\ipykernel_25476\3401446274.py:15: RuntimeWarning: divide by zero encountered in log
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
C:\Users\dalto\AppData\Local\Temp\ipykernel_25476\3401446274.py:15: RuntimeWarning: invalid value encountered in multiply
  term = np.where(y > 0, y * np.log(y / mu), 0.0)


SyntaxError: invalid syntax (<unknown>, line 1)